In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from avito_retrieval.data.loader import load_train
from avito_retrieval.retrieval.category_filter import CategoryFilter
from avito_retrieval.retrieval.hybrid import HybridRetriever
from avito_retrieval.metrics import recall_at_k
from avito_retrieval import config

train = load_train()

# Разбиваем по УНИКАЛЬНЫМ запросам, чтобы один и тот же запрос не оказался
# и в train_part (используется для CategoryFilter), и в val_part (валидация)
unique_queries = train["search_query"].unique()
train_queries, val_queries = train_test_split(unique_queries, test_size=0.2, random_state=config.SEED)

train_part = train[train["search_query"].isin(train_queries)]
val_part = train[train["search_query"].isin(val_queries)]

print(f"Train запросов: {len(train_queries)}, Val запросов: {len(val_queries)}")

Корпус после дедупа: 189212
Строю BM25-индекс: 5000 объявлений
Строю Dense-индекс: 5000 объявлений
DenseRetriever использует device: mps


Batches:  38%|███▊      | 15/40 [01:39<02:46,  6.64s/it]


KeyboardInterrupt: 

In [4]:
# Валидационный корпус: все объявления, встречающиеся в train_part + val_part,
# чтобы релевантные item_id гарантированно попадали в индекс
val_items = (
    pd.concat([train_part, val_part])
    .drop_duplicates(subset="item_id")
    [["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text", "item_category_id"]]
    .reset_index(drop=True)
)
print(f"Валидационный корпус: {len(val_items)} объявлений")

# Ground truth: search_query -> множество релевантных item_id
ground_truth = val_part.groupby("search_query")["item_id"].apply(set).to_dict()

# search_category для каждого запроса (берём первое встретившееся значение)
query_to_search_category = val_part.groupby("search_query")["search_category"].first().to_dict()

Корпус объявлений: 189212 (после дедупа)
Строю BM25-индекс: 189212 объявлений
Строю Dense-индекс: 189212 объявлений
DenseRetriever использует device: mps


Batches:   4%|▍         | 66/1479 [09:21<3:20:23,  8.51s/it]


KeyboardInterrupt: 

In [8]:
known_categories = set(val_items["item_category_id"].unique())

# top_n_categories тоже можно тюнить — начнём с 2, зафиксировав его отдельно от alpha
cat_filter = CategoryFilter(top_n_categories=2).fit(train_part, known_categories)

retriever = HybridRetriever(cat_filter, max_seq_length=128, batch_size=128)
retriever.build(val_items)  # строится один раз, дальше только retrieve_for_query с разными alpha

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (577 > 512). Running this sequence through the model will result in indexing errors


p50: 335.0 p95: 748.1499999999999 p99: 1192.09


In [9]:
alphas = np.arange(0.0, 1.01, 0.1)
results = {}

for alpha in alphas:
    predictions = {}
    for query in ground_truth.keys():
        search_category = query_to_search_category.get(query)
        predictions[query] = retriever.retrieve_for_query(
            search_query=query,
            search_category=search_category,
            alpha=alpha,
        )
    recall = recall_at_k(predictions, ground_truth, k=config.FINAL_TOP_K)
    results[round(alpha, 1)] = recall
    print(f"alpha={alpha:.1f} -> Recall@50={recall:.4f}")

best_alpha = max(results, key=results.get)
print(f"\nЛучший alpha: {best_alpha}, Recall@50={results[best_alpha]:.4f}")

max_seq_length=512: 161.8s для 3000 текстов
max_seq_length=256: 94.1s для 3000 текстов
max_seq_length=128: 40.9s для 3000 текстов


In [10]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(list(results.keys()), list(results.values()), marker="o")
plt.xlabel("alpha (1.0 = только BM25, 0.0 = только Dense)")
plt.ylabel("Recall@50")
plt.title("Recall@50 в зависимости от alpha (RRF)")
plt.grid(True, alpha=0.3)
plt.show()

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (521 > 512). Running this sequence through the model will result in indexing errors


title+params p50: 211.0
title+params p95: 684.0


In [ ]:
top_n_options = [1, 2, 3]
grid_results = {}

for top_n in top_n_options:
    for top_n in top_n_options:
        retriever.category_filter = CategoryFilter(top_n_categories=top_n).fit(train_part, known_categories)

    for alpha in alphas:
        predictions = {
            q: retriever.retrieve_for_query(q, query_to_search_category.get(q), alpha=alpha)
            for q in ground_truth.keys()
        }
        recall = recall_at_k(predictions, ground_truth, k=config.FINAL_TOP_K)
        grid_results[(top_n, round(alpha, 1))] = recall

best_combo = max(grid_results, key=grid_results.get)
print(
    f"Лучшая комбинация: top_n_categories={best_combo[0]}, alpha={best_combo[1]}, Recall@50={grid_results[best_combo]:.4f}")